# Feature Engineering & Modeling
In this notebook, we'll build features from the 30-day observation window and train machine learning models for Task A and Task B.


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, mean_absolute_error, make_scorer
from sklearn.preprocessing import LabelEncoder, StandardScaler
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')



## 1. Data Loading & Preprocessing


In [13]:
labels = pd.read_csv('train_labels.csv')
metadata = pd.read_csv('athlete_metadata.csv')
daily_activity = pd.read_csv('dailyActivity_merged.csv').rename(columns={'Id': 'athlete_id'})
sleep_day = pd.read_csv('sleepDay_merged.csv').rename(columns={'Id': 'athlete_id'})
training_sessions = pd.read_csv('training_sessions.csv')



In [14]:
# Feature Engineering Function
def filter_observation_window(df, date_col, id_col='athlete_id'):
    df = df.copy()
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
        start_dates = df.groupby(id_col)[date_col].min()
        df['day'] = (df[date_col] - df[id_col].map(start_dates)).dt.days + 1
        return df[df['day'] <= 30].copy()
    return df

def build_features(metadata, daily, sleep, training):
    df = metadata.copy()
    
    # Encode categoricals
    le = LabelEncoder()
    cat_cols = ['sport', 'gender', 'dominant_side', 'position']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
            df[col] = le.fit_transform(df[col])
            
    # Filter datasets to first 30 days (Observation Window)
    daily_obs = filter_observation_window(daily, 'ActivityDate')
    sleep_obs = filter_observation_window(sleep, 'SleepDay')
    training_obs = filter_observation_window(training, 'date')
    
    # Daily Activity Aggregations
    daily_agg = daily_obs.groupby('athlete_id').agg({
        'TotalSteps': ['mean', 'std', 'sum'],
        'TotalDistance': ['mean', 'max'],
        'VeryActiveMinutes': ['mean', 'sum'],
        'SedentaryMinutes': ['mean', 'std'],
        'Calories': ['mean', 'sum']
    })
    daily_agg.columns = ['_'.join(col).strip() for col in daily_agg.columns.values]
    daily_agg = daily_agg.reset_index()
    
    # Sleep Aggregations
    sleep_obs['SleepEfficiency'] = sleep_obs['TotalMinutesAsleep'] / sleep_obs['TotalTimeInBed']
    sleep_agg = sleep_obs.groupby('athlete_id').agg({
        'TotalMinutesAsleep': ['mean', 'std'],
        'SleepEfficiency': ['mean', 'min']
    })
    sleep_agg.columns = ['sleep_' + '_'.join(col).strip() for col in sleep_agg.columns.values]
    sleep_agg = sleep_agg.reset_index()
    
    # Training Sessions Aggregations
    training_obs['duration'] = training_obs['end_hour'] - training_obs['start_hour']
    training_obs['duration'] = training_obs['duration'].apply(lambda x: x if x >= 0 else x + 24)
    train_agg = training_obs.groupby('athlete_id').agg({
        'session_id': 'count',
        'duration': ['sum', 'mean']
    })
    train_agg.columns = ['train_' + '_'.join(col).strip() for col in train_agg.columns.values]
    train_agg = train_agg.reset_index()
    
    # Merge all
    df = pd.merge(df, daily_agg, on='athlete_id', how='left')
    df = pd.merge(df, sleep_agg, on='athlete_id', how='left')
    df = pd.merge(df, train_agg, on='athlete_id', how='left')
    
    # Fill NAs
    df = df.fillna(0)
    
    # Drop team_id (or could encode it)
    if 'team_id' in df.columns:
        df = df.drop(columns=['team_id'])
        
    return df

X_all = build_features(metadata, daily_activity, sleep_day, training_sessions)
data = pd.merge(X_all, labels, on='athlete_id', how='inner')

print(f"Features shape: {data.shape}")



Features shape: (3000, 31)


## 2. Task A: Classification (injured_in_risk_window)


In [15]:
X = data.drop(columns=['athlete_id', 'injured_in_risk_window', 'onset_day_offset', 'recovery_duration'])
y_clf = data['injured_in_risk_window']

X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)



In [16]:
# Grid Search for XGBoost
xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

scorer = make_scorer(f1_score)
grid_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=3, scoring=scorer, n_jobs=-1, verbose=1)
grid_xgb.fit(X_train_sc, y_train)

print(f"Best XGB Params: {grid_xgb.best_params_}")
preds_xgb = grid_xgb.predict(X_test_sc)
print(f"XGB F1 Score: {f1_score(y_test, preds_xgb):.4f}")



Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best XGB Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
XGB F1 Score: 0.6090


In [17]:
# CatBoost Classifier
cat_model = CatBoostClassifier(iterations=300, random_seed=42, verbose=0)
param_grid_cat = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1]
}

grid_cat = GridSearchCV(cat_model, param_grid_cat, cv=3, scoring=scorer, n_jobs=-1)
grid_cat.fit(X_train, y_train) # Catboost handles unscaled better but scaled is fine

print(f"Best CatBoost Params: {grid_cat.best_params_}")
preds_cat = grid_cat.predict(X_test)
print(f"CatBoost F1 Score: {f1_score(y_test, preds_cat):.4f}")



Best CatBoost Params: {'depth': 8, 'learning_rate': 0.05}
CatBoost F1 Score: 0.5872


In [18]:
# LightGBM Classifier
lgb_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)
param_grid_lgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, -1],
    'learning_rate': [0.01, 0.05, 0.1]
}

grid_lgb = GridSearchCV(lgb_model, param_grid_lgb, cv=3, scoring=scorer, n_jobs=-1)
grid_lgb.fit(X_train_sc, y_train)

print(f"Best LGBM Params: {grid_lgb.best_params_}")
preds_lgb = grid_lgb.predict(X_test_sc)
print(f"LGBM F1 Score: {f1_score(y_test, preds_lgb):.4f}")



[LightGBM] [Info] Number of positive: 840, number of negative: 1560
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000848 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3991
[LightGBM] [Info] Number of data points in the train set: 2400, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.350000 -> initscore=-0.619039
[LightGBM] [Info] Start training from score -0.619039
Best LGBM Params: {'learning_rate': 0.1, 'max_depth': -1, 'n_estimators': 100}
LGBM F1 Score: 0.6149


In [19]:
# Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
param_grid_rf = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(rf_model, param_grid_rf, cv=3, scoring=scorer, n_jobs=-1)
grid_rf.fit(X_train_sc, y_train)

print(f"Best RF Params: {grid_rf.best_params_}")
preds_rf = grid_rf.predict(X_test_sc)
print(f"RF F1 Score: {f1_score(y_test, preds_rf):.4f}")



Best RF Params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
RF F1 Score: 0.6163


## 3. Task B: Regression (Onset & Recovery)


In [20]:
# Filter only injured athletes for regression training
injured_data = data[data['injured_in_risk_window'] == 1]
X_reg = injured_data.drop(columns=['athlete_id', 'injured_in_risk_window', 'onset_day_offset', 'recovery_duration'])
y_onset = injured_data['onset_day_offset']
y_recovery = injured_data['recovery_duration']

# Train test split for regression
X_train_r, X_test_r, yo_train, yo_test, yr_train, yr_test = train_test_split(
    X_reg, y_onset, y_recovery, test_size=0.2, random_state=42
)

# Model for Onset Day (XGBoost Regressor)
reg_onset = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
reg_onset.fit(X_train_r, yo_train)
preds_onset = reg_onset.predict(X_test_r)
mae_onset = mean_absolute_error(yo_test, preds_onset)
print(f"Onset Day MAE: {mae_onset:.2f}")

# Model for Recovery Duration (LightGBM Regressor)
reg_recovery = lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
reg_recovery.fit(X_train_r, yr_train)
preds_recovery = reg_recovery.predict(X_test_r)
mae_recovery = mean_absolute_error(yr_test, preds_recovery)
print(f"Recovery Duration MAE: {mae_recovery:.2f}")



Onset Day MAE: 3.10
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000407 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3934
[LightGBM] [Info] Number of data points in the train set: 840, number of used features: 25
[LightGBM] [Info] Start training from score 11.490476
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

## 4. Final Pipeline Construction
Use the best classification model and best regression models to make predictions for a test set or for submission.


In [21]:
# Assuming 'grid_lgb' gave best F1 (for example)
best_classifier = grid_lgb.best_estimator_

# If we had test_data without labels:
# X_test_unseen = build_features(test_metadata, test_daily, test_sleep, test_training)
# test_preds_clf = best_classifier.predict(X_test_unseen)
# test_preds_onset = reg_onset.predict(X_test_unseen)
# test_preds_recovery = reg_recovery.predict(X_test_unseen)

# sample_submission['injured_in_risk_window'] = test_preds_clf
# sample_submission['onset_day_offset'] = np.clip(np.round(test_preds_onset), 1, 30)
# sample_submission['recovery_duration'] = np.clip(np.round(test_preds_recovery), 1, None)
# sample_submission.to_csv('submission.csv', index=False)

